# 05 — End-to-End Walkthrough

**Purpose:** A guided, user-oriented journey from raw pipeline artefacts through
database exploration, sequence retrieval, and a downstream result snapshot — all in
one cohesive story.

If you are new to PBI-Scope, run this notebook first.  It references individual notebooks
for deeper dives on each topic.

| | |
|---|---|
| **Expected inputs** | Pipeline logs at `/pipeline-logs`, database + FASTA files via `pbi.quick_connect()` |
| **Outputs** | Summary tables saved under `<results>/05_end_to_end_walkthrough/` |
| **Companion notebooks** | `00` logs · `01` database QC · `02` retrieval · `03` ML · `06` reproducibility |

## Story arc

1. [Setup & environment](#setup)
2. [Step 1 — Pipeline provenance check](#step1)
3. [Step 2 — Database at a glance](#step2)
4. [Step 3 — Query phage–host pairs](#step3)
5. [Step 4 — Retrieve sequences](#step4)
6. [Step 5 — Downstream analysis snapshot](#step5)
7. [Takeaways & next steps](#takeaways)


---
<a id='setup'></a>
## Setup

The setup cell below prints the `pbi` package version, the Python version, and the results directory used for exports.


In [1]:
import sys
import json
import os
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# Results directory
results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '05_end_to_end_walkthrough'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'pbi version  : {pbi.__version__}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Results dir  : {results_dir}')


pbi version  : 0.6.0
Python       : 3.10.21
Results dir  : /results/05_end_to_end_walkthrough


---
<a id='step1'></a>
## Step 1 — Pipeline Provenance Check

Before looking at data, confirm that the pipeline ran and understand *which*
data snapshot it consumed.  The provenance JSON written by the pipeline records
the snapshot date, the provider schema profile, and the build metadata.

See `00_pipeline_logs.ipynb` for a full exploration of all log artefacts.


In [2]:
logs_root = Path('/pipeline-logs')
prov_path = logs_root / 'csv' / 'pipeline_run_provenance.json'

if prov_path.exists():
    provenance = json.loads(prov_path.read_text())
    print('Pipeline run provenance')
    print('─' * 50)
    for key, val in provenance.items():
        print(f'  {key:<40} {val}')
else:
    provenance = {}
    print('ℹ️  pipeline_run_provenance.json not found.')
    print('   Run the Snakemake pipeline first, or mount ./pipeline_logs.')

# Quick sanity check on log directories
print()
for sub in ('logs', 'csv', 'reports'):
    d = logs_root / sub
    status = f'✅  {len(list(d.iterdir()))} files' if d.exists() else '❌  not found'
    print(f'  pipeline-logs/{sub}/  {status}')


Pipeline run provenance
──────────────────────────────────────────────────
  pipeline_run_timestamp                   2026-09-08T15:22:52Z
  provider_name                            PhageScope
  provider_release                         rolling
  provider_snapshot_date                   2026-07-05
  provider_schema_profile                  phagescope_v2
  provider_api_base_url                    https://phageapi.deepomics.org
  provider_provenance_mode                 config_pinned
  pbi_version                              
  git_commit                               
  download_records_count                   229

  pipeline-logs/logs/  ✅  14 files
  pipeline-logs/csv/  ✅  13 files
  pipeline-logs/reports/  ✅  12 files


---
<a id='step2'></a>
## Step 2 — Database at a Glance

`pbi.quick_connect()` auto-detects the optimised DuckDB database and all FASTA
indexes.  We use `get_stats()` for a one-shot overview.

Detailed QC and distribution plots live in `01_database_exploration.ipynb`.


In [3]:
retriever = pbi.quick_connect()
stats = retriever.get_stats()

print('Database overview')
print('─' * 50)
for category, values in stats.items():
    print(f'\n{category.upper()}')
    for key, value in values.items():
        if isinstance(value, (int, float)):
            print(f'  {key:<36} {value:,}')
        else:
            print(f'  {key:<36} {value}')


2026-09-09 19:41:43,139 - INFO - 📂 Checking FASTA index files:
2026-09-09 19:41:43,141 - INFO -    Phage index: True (82826.5 KB)
2026-09-09 19:41:43,142 - INFO -    Protein index: True (4376442.2 KB)
2026-09-09 19:41:43,144 - INFO - 📂 Loaded private phage mapping for 1 sources: ['test_private']
2026-09-09 19:41:43,144 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-09-09 19:41:43,149 - INFO -    Loaded mapping for 5363 hosts
2026-09-09 19:41:43,150 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-09-09 19:41:43,179 - INFO - 🔄 Starting background FASTA loading...
2026-09-09 19:41:43,180 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-09-09 19:41:43,180 - INFO - ✅ Initialization complete (FASTA loading in background)
2026-09-09 19:41:43,182 - INFO - ⏳ Waiting for FASTA loading to complete...
2026-09-09 19:41:52,542 - INFO -    ✅ Phage FASTA loaded in 9

Database overview
──────────────────────────────────────────────────

DATABASE
  phages                               1,350,641
  proteins                             71,971,209
  hosts                                5,527
  phage_host_associations              1,243,936
  source_breakdown                     [{'source_type': 'private', 'Source_DB': 'test_private', 'count': 5}, {'source_type': 'public', 'Source_DB': 'MetaVR', 'count': 225269}, {'source_type': 'public', 'Source_DB': 'GOV2', 'count': 195699}, {'source_type': 'public', 'Source_DB': 'MGV', 'count': 189680}, {'source_type': 'public', 'Source_DB': 'IMGVR', 'count': 177361}, {'source_type': 'public', 'Source_DB': 'GPD', 'count': 142809}, {'source_type': 'public', 'Source_DB': 'TemPhD', 'count': 66823}, {'source_type': 'public', 'Source_DB': 'ELGV', 'count': 56716}, {'source_type': 'public', 'Source_DB': 'CHVD', 'count': 44935}, {'source_type': 'public', 'Source_DB': 'URPC', 'count': 44633}, {'source_type': 'public', 'Source_D

---
<a id='step2b'></a>
## Step 2b — Private Data Duplicate Check

The pipeline automatically checks private phage sequences against public data
during database creation. Private phages with >99% identity to public sequences
are flagged as duplicates. This helps identify data that may have been derived
from old public database releases with modified IDs.

In [4]:
# Check for private/public duplicates
try:
    dup_count = retriever.conn.execute(
        "SELECT COUNT(*) FROM fact_phages WHERE is_duplicate_of_public = TRUE"
    ).fetchone()[0]
    
    total_private = retriever.conn.execute(
        "SELECT COUNT(*) FROM fact_phages WHERE source_type = 'private'"
    ).fetchone()[0]
    
    print(f'Private phages: {total_private:,}')
    print(f'Flagged as duplicates of public data: {dup_count:,}')
    
    if dup_count > 0:
        print('\nDuplicate details:')
        dups = retriever.conn.execute("""
            SELECT Phage_ID, Source_DB, duplicate_public_id, 
                   duplicate_pident, duplicate_qcovs
            FROM fact_phages
            WHERE is_duplicate_of_public = TRUE
            ORDER BY duplicate_pident DESC
        """).fetchdf()
        display(dups)
    else:
        print('No duplicates found between private and public data.')
except Exception as e:
    print(f'Could not check duplicates: {e}')
    print('The duplicate columns may not exist if the database was built')
    print('before the duplicate detection feature was added.')

Private phages: 5
Flagged as duplicates of public data: 0
No duplicates found between private and public data.


---
<a id='step3'></a>
## Step 3 — Query Phage–Host Pairs

We retrieve a small, tractable set of phage–host pairs to work with
through the rest of this walkthrough.  The query filters to *lytic* phages
linked to *Escherichia coli* hosts that have a downloadable assembly — a
biologically meaningful and data-complete subset.

Adjust the filter to suit your research question.


In [5]:
pairs_df = retriever.query_phage_host_pairs(
    phage_filters={'Lifestyle': 'virulent'},
    limit=20,
)
pairs_df

2026-09-09 19:54:08,195 - INFO - 🔍 Querying phage-host pairs...
2026-09-09 19:54:08,746 - INFO - 📊 Found 20 phage-host pairs
2026-09-09 19:54:08,747 - INFO - 📥 Fetching sequences for 20 phages and 14 unique hosts
2026-09-09 19:54:13,829 - WARNING - ⚠️  Removed 7 pairs with missing sequences (0 phages missing, 4 hosts missing)
2026-09-09 19:54:13,830 - WARNING -    Missing host IDs (sample): GCF_058182505_1, GCF_059731855_1, GCF_976985775_1, GCF_987193625_1
2026-09-09 19:54:13,831 - INFO - ✅ Retrieved 13 complete phage-host pairs with sequences


,Phage_ID,Host_ID,Phage_Source,Phage_Source_Type,Phage_Length,Phage_GC,Phage_Taxonomy,Phage_Completeness,Phage_Lifestyle,Phage_Cluster,Phage_Subcluster,Species_Name,Host_Assembly_Level,Host_Length,Host_GC,Host_RefSeq_Category,Phage_Sequence,Host_Sequence
0,IMGVR_UViG_3300045988_156133|3300045988|Ga0495776_094719,GCF_976985875_1,IMGVR,public,42997,53.008349,Caudoviricetes,Medium-quality,virulent,cluster_390232,subcluster_471493,Phocaeicola vulgatus,Complete Genome,5230292,42.59,na,CTGAATGTTTTTGTTTTTCTGCGGAAAGAAAAATATGGAAGAAGGAAATGGCTGTG...,GAGAGACAGGCTCTCGAACCTATACATTCAAGAAATATCATAAAATTGCAAATTAC...
3,IMGVR_UViG_3300045988_174244|3300045988|Ga0495776_185018,GCF_059738785_1,IMGVR,public,46845,35.214004,Caudoviricetes,High-quality,virulent,cluster_130710,subcluster_157547,Streptococcus nakanonensis,Complete Genome,2127737,40.36,reference genome,AATGCTAATCTTCGTCGTTTTACTCCTTGACTAGCAAACTTACCGCCTCAACATGT...,AACATTGTGGATTATTTTTCACAGCTTGTGGAAAATTCTTGTTTTCTATGGTAAAA...
6,IMGVR_UViG_3300045988_062769|3300045988|Ga0495776_005154,GCF_055398035_1,IMGVR,public,100860,33.303589,Crassvirales,Complete,virulent,cluster_317694,subcluster_383867,Bacteroides reticulotermitis,Complete Genome,5470359,43.26,reference genome,ACTTAGTTGGTTGGGCTGCTGGTTGCATCAGTAAAACTATAATTAAAGAGAAACAA...,TGCGATTCATCAACAAATAAATGGTCGATGCCCATCAGCCTGAAATCCACCGCATC...
8,IMGVR_UViG_3300045988_052728|3300045988|Ga0495776_014876,GCF_976985875_1,IMGVR,public,182061,36.845343,Caudoviricetes,Complete,virulent,cluster_423226,subcluster_510758,Phocaeicola vulgatus,Complete Genome,5230292,42.59,na,ATGTAGATTATGAAGAAGTAGAGGGTTTAGACGATTTATTTGCTGAATGACATACT...,GAGAGACAGGCTCTCGAACCTATACATTCAAGAAATATCATAAAATTGCAAATTAC...
9,IMGVR_UViG_3300045988_065473|3300045988|Ga0495776_016290,GCF_052659795_1,IMGVR,public,58410,48.707413,Caudoviricetes,Complete,virulent,cluster_39887,subcluster_48675,Faecalibacterium prausnitzii,Complete Genome,2804975,56.59,na,CTGCACAAAAATCAATCTCCCCAGCACACGGAATCCGCTTCGTCTGGCCGCTGCCA...,ATGGATTCTTTCAAGGACGTTTTAGAGGCCGCCCAGGCATACTGCAAAACGCAGAT...
10,IMGVR_UViG_3300045988_093152|3300045988|Ga0495776_019922,GCF_052659795_1,IMGVR,public,62487,51.117833,Caudoviricetes,Complete,virulent,cluster_263731,subcluster_318436,Faecalibacterium prausnitzii,Complete Genome,2804975,56.59,na,AGCTTGCATAATGTGCTGTATTGTCTCGTTTGAAATTCCGCTCCAAGTCATAGCCT...,ATGGATTCTTTCAAGGACGTTTTAGAGGCCGCCCAGGCATACTGCAAAACGCAGAT...
12,IMGVR_UViG_3300045988_055490|3300045988|Ga0495776_061151,GCF_041014885_1,IMGVR,public,32892,53.785115,Caudoviricetes,High-quality,virulent,cluster_250306,subcluster_302331,Bacteroides thetaiotaomicron,Complete Genome,6018559,43.09,na,ATTATACCAGATATCGGGACGAAGCGCAAGTAAAAAAGTGATGGAACATCCGGCCT...,ATGATTGAATCAAATCATGTTGTACTTTGGAACCGCTGTCTCGAAGTAATTAAAGA...
13,IMGVR_UViG_3300045988_169166|3300045988|Ga0495776_096064,GCF_051861815_1,IMGVR,public,88014,24.973300,Crassvirales,High-quality,virulent,cluster_344863,subcluster_416520,Segatella copri,Complete Genome,4246773,44.76,na,AGATATTTATATATTAATAATATATAAATACTTCTTACGCGCGAGCGCGCGCACGT...,CCCCTATGACAATACAACTCCATCCCCAGTGTTTACTGGGGATACAGAGTATTTTA...
14,IMGVR_UViG_3300045988_034081|3300045988|Ga0495776_185026,GCF_055390445_1,IMGVR,public,39039,45.234253,Caudoviricetes,Medium-quality,virulent,cluster_115137,subcluster_139250,Prevotella denticola,Complete Genome,3170192,49.95,reference genome,GTACTATTTTGCCGAATAAACGTACTTTCATTTCAAAGAATAAAAAAAGTGCAGAA...,ATTACCGCTTATCCCCACCTATCCCCCAGCATGAACTTCCCCTTTTACATCGCCCG...
15,IMGVR_UViG_3300045988_050960|3300045988|Ga0495776_011455,GCF_982303505_1,IMGVR,public,99855,32.847629,Caudoviricetes,Complete,virulent,cluster_347353,subcluster_419587,Staphylococcus aureus,Complete Genome,2846193,32.86,na,ACTTTTTTGATTTACACTTTATATCATTTACTATCTATGAATTATTATCAGTTAGG...,ATGTCGGAAAAAGAAATTTGGGAAAAAGTGCTTGAAATTGCTCAAGAAAAATTATC...


In [6]:
pairs_df = retriever.query_phage_host_pairs(
    phage_filters={'Lifestyle': 'virulent'},
    host_filters={'Organism_Name': '%Escherichia%'},
    limit=20,
)

print(f'Pairs returned: {len(pairs_df)}')

if not pairs_df.empty:
    # Show a tidy subset of columns if they exist
    preview_cols = [c for c in [
        'Phage_ID', 'Phage_Name', 'Lifestyle', 'Genome_Length',
        'GC_Content', 'Host_Assembly_Accession', 'Organism_Name'
    ] if c in pairs_df.columns]
    display(pairs_df[preview_cols].head(10))
else:
    print('No pairs found — try relaxing the filter or check the database.')


2026-09-09 19:54:17,332 - INFO - 🔍 Querying phage-host pairs...
2026-09-09 19:54:17,605 - INFO - 📊 Found 20 phage-host pairs
2026-09-09 19:54:17,606 - INFO - 📥 Fetching sequences for 20 phages and 1 unique hosts
2026-09-09 19:54:17,729 - INFO - ✅ Retrieved 20 complete phage-host pairs with sequences


Pairs returned: 20


,Phage_ID
0,uvig_314890
1,uvig_319593
2,uvig_325105
3,uvig_326817
4,uvig_334187
5,uvig_337061
6,uvig_341904
7,uvig_342304
8,uvig_353783
9,uvig_376294


In [7]:
# Save the pairs table for later reference
if not pairs_df.empty:
    out_path = results_dir / 'lytic_ecoli_pairs_sample.csv'
    pairs_df.to_csv(out_path, index=False)
    print(f'Saved {len(pairs_df)} rows → {out_path}')


Saved 20 rows → /results/05_end_to_end_walkthrough/lytic_ecoli_pairs_sample.csv


---
<a id='step4'></a>
## Step 4 — Retrieve Sequences

For the first few pairs we now pull the actual DNA sequences.  Because FASTA
files are pyfaidx-indexed, each retrieval is a direct byte-offset read — there
is no need to scan the whole file.

> **Note:** Some pairs may not have a host genome downloaded (host assembly exists
> in the database but the FASTA was not fetched).  These are silently skipped.
> See `00_pipeline_logs.ipynb § 4` for how to inspect download failures.


In [8]:
MAX_PAIRS = 5  # Keep this small for the walkthrough

retrieved = []
if not pairs_df.empty:
    phage_ids = pairs_df['Phage_ID'].dropna().unique().tolist()
    for phage_id in phage_ids[:MAX_PAIRS]:
        try:
            seq = retriever.get_phage_sequence(phage_id)
            if seq:
                retrieved.append({'Phage_ID': phage_id, 'Seq_length': len(seq), 'GC': round(
                    (seq.count('G') + seq.count('C')) / len(seq) * 100, 2
                ) if seq else None})
                print(f'  {phage_id:<30}  {len(seq):>8,} bp')
        except Exception as exc:
            print(f'  {phage_id:<30}  ⚠️  {exc}')

if not retrieved:
    print('No sequences retrieved — FASTA files may not be available in this environment.')


  uvig_314890                       20,879 bp
  uvig_319593                       21,407 bp
  uvig_325105                       12,714 bp
  uvig_326817                       14,029 bp
  uvig_334187                       45,429 bp


---
<a id='step5'></a>
## Step 5 — Downstream Analysis Snapshot

Even a small retrieval is enough to compute simple bioinformatic summaries:
genome length distribution and GC content, two of the most basic phage
characterisation metrics.  The `03_ml_streaming.ipynb` notebook builds on
these features for a full ML workflow.


In [9]:
# ── Genome length & GC distribution from the pair query ─────────────────
if not pairs_df.empty:
    cols_to_plot = [c for c in ['Genome_Length', 'GC_Content'] if c in pairs_df.columns]
    if cols_to_plot:
        fig, axes = plt.subplots(1, len(cols_to_plot), figsize=(6 * len(cols_to_plot), 4))
        if len(cols_to_plot) == 1:
            axes = [axes]
        for ax, col in zip(axes, cols_to_plot):
            pairs_df[col].dropna().hist(ax=ax, bins=20, edgecolor='k', alpha=0.7)
            ax.set_title(col.replace('_', ' '))
            ax.set_xlabel(col)
            ax.set_ylabel('Count')
        plt.suptitle('Lytic E. coli-infecting phages — sample distribution', y=1.02)
        plt.tight_layout()
        fig_path = results_dir / 'sample_distributions.png'
        plt.savefig(fig_path, dpi=96, bbox_inches='tight')
        plt.show()
        print(f'Figure saved → {fig_path}')
    else:
        print('Genome_Length / GC_Content columns not present in this query result.')
else:
    print('No data to plot.')


Genome_Length / GC_Content columns not present in this query result.


---
<a id='takeaways'></a>
## Takeaways & Next Steps

This walkthrough demonstrated the full PBI-Scope data path in five steps:

| Step | What happened |
|------|---------------|
| 1 | Verified the pipeline ran and identified the data snapshot |
| 2 | Got a one-screen database summary (counts, coverage) |
| 3 | Queried a biologically meaningful phage–host subset |
| 4 | Retrieved phage DNA sequences with O(1) FASTA access |
| 5 | Computed basic genome statistics as a downstream result |

### Where to go next

| Goal | Notebook |
|------|----------|
| Inspect all pipeline logs in depth | `00_pipeline_logs.ipynb` |
| Full database QC and distribution plots | `01_database_exploration.ipynb` |
| Exhaustive sequence retrieval API | `02_sequence_retrieval.ipynb` |
| ML feature engineering and training | `03_ml_streaming.ipynb` |
| Release consumer / Zenodo download | `04_data_release_exploration.ipynb` |
| Reproducibility and provenance deep-dive | `06_reproducibility.ipynb` |
| GFF3 gene annotations | `07_gff3_annotations.ipynb` |
| REST API access | `08_api_client.ipynb` |
| Sequence similarity (BLAST) search | `09_blast_search.ipynb` |


In [10]:
# Clean up
#retriever.close()
#print('Database connection closed.')
